In [ ]:
!pip install PyPDF2 -q
!pip install pdf2image -q
!pip install openai -q
!pip install PyYAML -q
!pip install tiktoken -q
!pip install pycryptodome -q
!pip install reportlab -q
!pip install pymupdf
!pip install langdetect

In [ ]:
!sudo apt-get update
!sudo apt-get install -y poppler-utils


Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:5 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
You might want to run 'apt --fix-broken install' to correct these.
The following packages have

# 0. Librerias

In [ ]:
import os
import csv
import re
import shutil
from datetime import datetime
import yaml
from io import BytesIO
import pandas as pd
from PyPDF2 import PdfReader, PdfWriter
from pdf2image import convert_from_bytes
import time
import json
import base64
import openai
from reportlab.lib.pagesizes import letter
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.platypus import SimpleDocTemplate, Paragraph
import fitz
from openai import OpenAI
from typing import Tuple, Dict, Optional, List
import math
from collections import Counter
import re
from langdetect import detect, DetectorFactory
from langdetect.lang_detect_exception import LangDetectException

# Fijar semilla para resultados reproducibles (importante para el TFG)
DetectorFactory.seed = 42

# 1. Funciones de Utilidad

In [ ]:
def load_config(config_file): # Parameterize config file name
    """
    Carga la configuración desde el fichero YAML.
    """
    try:
        with open(config_file, "r") as f:
            config = yaml.safe_load(f)
        print("Configuración cargada exitosamente.")
        return config
    except Exception as e:
        print(f"Error al cargar la configuración: {e}")
        raise


# Consideraciones del procesamiento.

Aviso con caracter académico: El método onlyimages con GPT-4o solo sería necesario si los documentos fueran escaneos (imágenes incrustadas en PDF). Para documentos nativos como este, usarlo sería un gasto innecesario y además introducirías una dependencia externa en una etapa trivial del pipeline, lo que debilitaría la justificación metodológica en la memoria.

Estrategia Revisada: Pipeline Híbrido
La conclusión es clara: necesitas un pipeline híbrido inteligente que detecte automáticamente el tipo de documento y aplique el método adecuado. Esto es además mucho más interesante para el TFG, ya que justifica una decisión de diseño arquitectónico con evidencia empírica.

Mejoras que debo implemtar al código de muestra:

1-Cambio de formato de salida: En lugar de generar un PDF de salida con el texto (lo cual es redundante), tu versión guardará el texto extraído directamente en formato .txt o .json por documento. Esto es más eficiente para la etapa posterior de chunking y vectorización.

2-Uso de la librería de PyMuPDF

3-Pipeline híbrido para poder hacer extracción del texto de los power points y los pdfs con imágenes.

# Pipeline a Seguir para la Extracción de los Datos


Documentos → Páginas → Detección por página → Extracción del texto
(pymupdf | OpenAI Vision) → Markdown (opcional) → JSON → Disco


La clave del diseño es que la unidad de procesamiento es la página, no el documento. Eso es más granular, más eficiente en coste y más trazable.

Además los logs podrían ser prescindibles, no tan relvantes como en el pipeline que llevó a cabo la empresa.


1. ¿Es viable la detección página a página?
Sí, completamente viable y es la estrategia óptima. El enfoque es simple: intentas extraer el texto con pymupdf primero, mides los caracteres obtenidos, y solo llamas a la API si el resultado está por debajo de un umbral. El coste de "intentar con pymupdf" es prácticamente cero (es local, sin red, sin latencia), así que no pierdes nada probándolo siempre primero.
El umbral de caracteres que se use para distinguir el tipo de procesamiento, es un hiperparámetro que deberás justificar empíricamente en la memoria. Documenta qué páginas caen por debajo y comprueba que efectivamente son páginas visuales. Si tienes páginas con solo un título (50 chars) que son en realidad texto puro, el umbral falla. Podrías calibrarlo revisando una muestra del corpus manualmente.

2. ¿Es necesario convertir a Markdown?
No es estrictamente necesario, pero sí aporta valor real si tu corpus tiene estructura. La recomendación práctica para tu TFG: Aplica Markdown solo para aquellos documentos que contengan páginas procesadas por OpenAI Vision. Se haría una vez tenemos el texto ya gaurdado en el json. Para los documentos con páginas procesadas con pymupdf, limpieza básica es suficiente. Así obtienes el beneficio donde importa sin añadir un paso extra de conversión para todos los documentos.

De esta forma tendríamos a priori texto plano en todo, y solo se pasaría a Markdown en una única pasada final, a aquellos documentos que comprobásemos a posteriori que al menos una de sus páginas fue extraida con la API.
Tanto pymupdf como Vision extraen solo texto plano limpio. Una vez tienes el texto_completo concatenado, haces una única llamada a la API para convertir todo el documento a Markdown.

Ventaja: El documento resultante es coherente, uniforme y la conversión Markdown tiene contexto completo para detectar correctamente las cabeceras, listas y estructura real del documento.

Coste real: Una llamada extra por documento, pero solo para los documentos que tengan al menos una página procesada por Vision. Los documentos 100% pymupdf no necesitan esa llamada.


3. ¿Es viable el JSON? ¿Por páginas o bloque único?
El JSON es el formato correcto para esta fase. Es legible, parseble, y perfecto para añadir metadatos. Sobre la estructura, recomiendo firmemente el enfoque por páginas por tres razones:

i).Trazabilidad en el RAG: Cuando el sistema recupera un chunk, puedes decirle al usuario "esta información viene de la página 7 del documento X". Eso es fundamental para la evaluación de fidelidad con RAGAS.

ii).Registro del método de extracción: Sabes qué páginas usaron Vision y cuáles pymupdf, lo que es valioso para el análisis de resultados del TFG.

iii).Flexibilidad en el chunking posterior: Puedes decidir hacer chunking respetando los límites de página o ignorándolos, según convenga.


Ejemplo del que imagino que sea mi json final:

{
  "nombre_documento": "CodigoEtico.pdf",
  "num_paginas": 14,
  "fecha_procesado": "2026-02-20",
  "paginas": [
    {
      "num_pagina": 1,
      "metodo_extraccion": "pymupdf",
      "texto_crudo": "Código Ético AMC-FTN-CE-13..."
    },
    {
      "num_pagina": 2,
      "metodo_extraccion": "openai_vision",
      "texto_crudo": "Nuestra Misión AMC Natural Drinks..."
    }
  ],
  "texto_completo": "Código Ético AMC...[texto concatenado de todas las páginas]",
  "texto_completo_markdown": "# Código Ético\n\n## Nuestra Misión..." (opcional)
}

```
Para cada documento:
│
├── Por cada página:
│   ├── pymupdf → is_text_usable() → True  → texto plano
│   └── pymupdf → is_text_usable() → False → Vision API → texto plano
│
├── Concatenar todas las páginas → texto_completo
│
└── Guardar JSON:
    - metadatos del documento
    - array de páginas (texto + método)
    - texto_completo (bloque único, texto plano)
    - Opcional o si fuese mejor para el chunking --> pasar a formato Markdown.

```

## Función para determinar si la página debe ser procesada por pymupdf o vía prompting a la API de OpenAI.


Criterio 5 — Entropía de Shannon
La entropía mide la aleatoriedad del texto. Texto real en español tiene una entropía entre ~3.5 y ~5.0 bits/carácter. Por debajo es texto repetitivo (headers, basura repetida), por encima es texto aleatorio (garbage de codificación).


Criterio 6 — Ratio de Palabras Alfabéticas
De todas las "palabras" separadas por espacios, ¿qué porcentaje contiene al menos una letra? Texto basura de codificación tiene muchos tokens que son solo números, símbolos o combinaciones sin letras.



Criterio 8 - Tablas mal extraídas. Este criterio es especialmente relevante para justificar el pipeline híbrido. Documenta que la detección de tablas fragmentadas fue identificada empíricamente sobre el corpus de AMC, y que el umbral del 70% fue calibrado para no penalizar páginas con listas legítimas de ítems cortos.


| Criterio               | Tipo de basura que detecta                        | Ejemplo                        |
| ---------------------- | ------------------------------------------------- | ------------------------------ |
| Entropía < 2.5         | Texto hiperrepetitivo                             | "aaaaaaaa bbbbb aaaa"          |
| Entropía > 6.0         | Garbage aleatorio de codificación rota            | "ÿþ\\x00ýóÄ\\x12öü"            |
| Alpha word ratio < 50% | Tokens sin letras, tablas de números sin contexto | "123 456 789 0.45 3.2"         |
| Most common char > 20% | Un símbolo dominante repetido                     | "////// texto //// texto ////" |

In [ ]:


def shannon_entropy(text: str) -> float:
    """Calcula la entropía de Shannon en bits por carácter."""
    if not text:
        return 0.0
    freq = Counter(text)
    length = len(text)
    return -sum(
        (count / length) * math.log2(count / length)
        for count in freq.values()
    )

def is_text_usable(text: str, min_chars: int = 40) -> bool:

    """
    Evalúa si el texto extraído por pymupdf es de calidad suficiente
    para ser usado directamente, o si debe procesarse vía Vision.

    Criterios:
      1. Longitud mínima
      2. Ratio de caracteres válidos (alfanuméricos + puntuación común)
      4. Ratio de espacios (texto real tiene ~10-20% de espacios)
      5. Entropía de Shannon (detecta texto aleatorio o extremadamente repetitivo)
      6. Ratio de palabras alfabéticas (detecta tokens sin letras)
      7. Ratio del carácter más frecuente (detecta secuencias repetidas)
    """

    # Criterio 1: Si la página está vacía probablemente fuese un escaneo
    if not text or len(text.strip()) < min_chars:
        return False

    # Criterio 2: ratio de caracteres válidos
    # Texto basura suele tener muchos caracteres raros/de control
    valid_chars = sum(
        1 for c in text
        if c.isalnum() or c in ' .,;:!?-()\n\t"\'áéíóúüñÁÉÍÓÚÜÑ'
    )
    valid_ratio = valid_chars / len(text)
    if valid_ratio < 0.85:
        return False


    # Criterio 4: ratio de espacios
    # Menos del 5% de espacios → probablemente texto sin separación real
    space_ratio = text.count(' ') / len(text)
    if space_ratio < 0.05:
        return False

    #--- Criterio 5: entropía de Shannon ---
    # Texto real en español/inglés: entre 3.5 y 5.5 bits/char
    # Por debajo → texto repetitivo/basura, por encima → aleatorio/codificación rota
    entropy = shannon_entropy(text)
    if entropy < 2.5 or entropy > 6.0:
        return False

    # --- Criterio 8: Para cuando la tabla no se detectó correctamente; Ratio de líneas de una sola palabra ---
    # Tablas mal extraídas por pymupdf producen una palabra/número por línea.
    # Si más del 70% de las líneas no vacías tienen una única palabra, es indicativo de tabla fragmentada
    # UNIDAD        ← 1 token por línea
    # UNIDADES      ← 1 token por línea
    # ENVASE        ← 1 token por línea
    # 2             ← 1 token por línea
    # 105           ← 1 token por línea
    # 17,50         ← 1 token por línea

    non_empty_lines = [line.strip() for line in text.splitlines() if line.strip()]
    if len(non_empty_lines) > 5:  # Solo aplicar si hay suficientes líneas para ser significativo
        single_token_lines = sum(
            1 for line in non_empty_lines
            if len(line.split()) == 1
        )
        single_token_ratio = single_token_lines / len(non_empty_lines)
        if single_token_ratio > 0.70:
            return False


    return True


## Función para escribir a disco el json final generado.

In [ ]:
def write_json_to_disk(output_directory: str, file_content: dict) -> str:
    """
    Guarda el contenido del documento procesado como fichero JSON en disco.
    Retorna la ruta del fichero generado.
    """
    file_name = file_content["nombre_procesado"]
    output_path = os.path.join(output_directory, f"{file_name}.json")

    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(file_content, f, ensure_ascii=False, indent=4)

    print(f" JSON guardado: {output_path}")
    return output_path


## Procesamirento adicional


**Lo que No Vale la Pena**


Cualquier limpieza más agresiva (OCR correction, corrección ortográfica automática, detección de idioma página a página) añade complejidad sin beneficio demostrable para un corpus de documentos corporativos de calidad media-alta como el de AMC. Si el tribunal pregunta, la respuesta es siempre la misma: el coste de complejidad no está justificado por la mejora esperada en las métricas de evaluación del RAG.

Por ejemplo: Para el corpus concreto de AMC te añadiría una comprobación adicional: los documentos legales como el BOE y las políticas internas suelen tener números de artículo como líneas aisladas ("Artículo 1.", "1.", "1.1"). Esos no los queremos eliminar — y la restricción de 1-3 dígitos en remove_isolated_page_numbers ya los protege correctamente.

Dado que se tratan de documentos corporativos que están muy bien revisados ortográficamente no es necesario aplicar ningún tipo de corrección ortográfica como si de texto de RRSS se tratase (por hacer analogía a lo visto en la asignatura de tercero de PLN).


¿Mantener los Índices?
Sí, mantenlos. El índice de un documento contiene el vocabulario estructural del mismo — nombres de secciones, términos clave — y es información válida para el RAG. Cuando el sistema busque por similitud semántica, chunks del índice pueden ser relevantes para preguntas del tipo "¿qué temas cubre este informe?". Eliminarlo sería descartar contexto útil.


¿correos electrónicos?
¿páginas web?
No los elimines, transfórmalos. Un email como privacidad@amcglobal.com o una URL como www.amcglobal.com son datos de contacto reales que un usuario del RAG podría necesitar. Lo que sí debes evitar es que el tokenizador los trate como tokens raros. La solución es normalizarlos con un placeholder. Poner el placeholder indicando que lo que viene a continuación es la url o el email, permite al RAG identificarlo con mayor precisión, sin perder información.


Función normalize_special_charts: Cuando un diseñador crea un documento en Word, InDesign o PowerPoint, el procesador de texto sustituye automáticamente ciertos caracteres ASCII por sus equivalentes tipográficos mejorados. Es lo que se llama "smart typography" o tipografía inteligente.
Cuando el PDF se genera, esos caracteres tipográficos se embeben tal cual en el fichero. pymupdf los extrae fielmente, devolviendo exactamente lo que hay en el PDF: los codepoints Unicode tipográficos, no los ASCII equivalentes.

Por Qué Normalizarlos
Tres razones concretas para tu pipeline:

1. Consistencia en el embedding: – y - son tokens distintos para el tokenizador. Una búsqueda semántica sobre "páginas 10-20" no encontrará "páginas 10–20" si no están normalizados.

2. Compatibilidad: Algunos sistemas de chunking, bases de datos vectoriales o herramientas de evaluación como RAGAS pueden comportarse de forma inesperada con caracteres fuera del rango ASCII básico.

3. Legibilidad del Golden Dataset: Cuando construyas manualmente las preguntas y respuestas de referencia, escribirás - y " desde el teclado. Si el corpus tiene – y ", las comparaciones de cadenas fallarán silenciosamente.

El non-breaking space (\u00a0) merece una mención especial: es visualmente idéntico al espacio normal pero es un codepoint diferente. Los documentos corporativos lo usan para evitar que ciertos textos se partan entre líneas (ej. AMC\u00a0Global siempre aparece junto). Si no lo normalizas, re.sub(r'[\t ]+', ' ', text) no lo detectará porque [\t ] solo captura el espacio ASCII \u0020, no el \u00a0.


U+002D  →  -   (hyphen-minus, el del teclado)
U+2013  →  –   (en dash, más largo)
U+2014  →  —   (em dash, el más largo)
Los tres son visibles, los tres son caracteres reales. Lo que ocurre es que tu teclado físico solo tiene acceso directo a U+002D, mientras que los procesadores de texto como Word insertan automáticamente U+2013 o U+2014 según el contexto.

Por Qué Esto Importa Para tu Pipeline
Cuando pymupdf lee el PDF y te devuelve —, no está haciendo ninguna transformación rara — te está devolviendo exactamente el carácter que el diseñador puso en el documento. El problema no es pymupdf, es que ese carácter perfectamente válido y visible es diferente a nivel de codepoint del guión que tú escribirías desde el teclado, y esa diferencia invisible para el ojo humano es muy visible para un tokenizador o para una comparación de strings en Python:

python
'–' == '-'   # False — aunque visualmente sean casi idénticos
len('…')     # 1 — un solo carácter, aunque parezcan tres puntos
len('...')   # 3 — tres caracteres ASCII
Ese último ejemplo es especialmente importante: … es un único token para el tokenizador, mientras que ... son tres. Esa diferencia puede afectar sutilmente a cómo el modelo interpreta el texto.

Para referirnos a los caracteres que queremos normalizar, en python podemos referirnos a ellos directamente por su codepoint. En Python un string es simplemente una secuencia de codepoints Unicode, así que podemos referirnos a cualquier carácter por su codepoint con la notación \uXXXX y Python lo trata como el carácter en sí.

Unicode reserva ese bloque entero (Private Use Area, PUA) para que los fabricantes de fuentes asignen glifos propios sin estandarizar. Es exactamente lo que hace la fuente de ese PDF con su bullet point — lo almacena en un codepoint del PUA

In [ ]:
import re

def limpiar_texto_completo(texto: str) -> str:
    # 1. Eliminar líneas que son solo un número de sección (11. o 6.2.2.)
    texto = re.sub(r'^\s*\d+(\.\d+)*\.\s*$', '', texto, flags=re.MULTILINE)
    # 2. Eliminar líneas de índice con puntos de relleno (TÍTULO.......N)
    texto = re.sub(r'^.+\.{4,}\s*\d+\s*$', '', texto, flags=re.MULTILINE)
    # 3. Colapsar saltos de línea múltiples que queden tras las eliminaciones
    texto = re.sub(r'\n{3,}', '\n\n', texto)
    return texto.strip()


def strip_markdown_codeblock(text: str) -> str:
    """
    Elimina bloques de código markdown si el modelo los añade incorrectamente.
    Cubre: ```markdown ... ``` y ``` ... ```
    """
    text = text.strip()
    # Patrón: apertura opcional con lenguaje (```markdown, ```plaintext, ```) → contenido → cierre (```)
    match = re.match(r'^```[a-zA-Z]*\n(.*?)```$', text, flags=re.DOTALL)
    if match:
        return match.group(1).strip()
    return text

def normalize_urls_and_emails(text: str) -> str:
    """
    Reemplaza URLs y emails por placeholders legibles preservando el valor.
    Esto permite preservar la información real del email o url, a la vez que lo marca con una
    etiqueta de tipo de entidad para que a posteriori al sistema RAG le sea más sencillo
    recuperarlo ante una pregunta que busque ese tipo de datos.
    """
    # Emails — grupo de captura \1 para preservar el valor
    text = re.sub(
        r'([a-zA-Z0-9._%+\-]+@[a-zA-Z0-9.\-]+\.[a-zA-Z]{2,})',
        r'[EMAIL: \1]',
        text
    )
    # URLs — grupo de captura \1 para preservar el valor
    text = re.sub(
        r'(https?://\S+|www\.\S+)',
        r'[URL: \1]',
        text
    )
    return text


def normalize_whitespace(text: str) -> str:
    #Eliminamos tabulaciónm excesiva y espacios múltiples
    text = re.sub(r'[\t ]+', ' ', text)
    # text = re.sub(r'\n{3,}', '\n\n', text)
    text = re.sub(r'(\n[ \t]*){3,}', '\n\n', text)
    return text

def remove_isolated_page_numbers(text: str) -> str:
    """
    Elimina líneas que solo contienen un número de 1-3 dígitos.
    Restringe a máximo 3 dígitos para no eliminar años, importes,
    u otros datos numéricos legítimos en contexto.
    """
    return re.sub(r'^\s*\d{1,3}\s*$', '', text, flags=re.MULTILINE)

def normalize_special_chars(text: str) -> str:
    """
    Normaliza caracteres tipográficos especiales a sus equivalentes ASCII.
    Incluye guiones, comillas, elipsis, bullets y espacios no estándar.
    """
    replacements = {
        # Guiones tipográficos
        '\u2013': '-', '\u2014': '-',
        # Comillas tipográficas dobles
        '\u201c': '"', '\u201d': '"',
        # Comillas tipográficas simples
        '\u2018': "'", '\u2019': "'",
        # Puntos suspensivos
        '\u2026': '...',  # …
        # Bullets y listas
        '\u2022': '-', '\u00b7': '-',
        # Espacios no estándar
        '\u00a0': ' ', '\u200b': '',
        #Glifo que pymupdf no consigue mapear a ningún codep
        '\ufffd': '-'
    }
    for char, replacement in replacements.items():
        text = text.replace(char, replacement)

    # Eliminar cualquier carácter del área de uso privado (Private Use Area)
    # que no haya sido capturado individualmente
    text = re.sub(r'[\ue000-\uf8ff]', '-', text)
    return text

def clean_page_text(text: str) -> str:
    """Aplica todos los procesamientos de limpieza en el orden correcto."""
    #text = strip_markdown_codeblock(text)       # 0. Por si el modelo no obedeciese a la instrucción de evitar
    text = remove_isolated_page_numbers(text)    # 1. Números de página aislados
    text = normalize_special_chars(text)         # 2. Caracteres especiales
    text = normalize_whitespace(text)            # 3. Espacios y saltos de línea
    text = normalize_urls_and_emails(text)       # 4. Señalizar URLs y emails
    text = limpiar_texto_completo(text)          # 5. Eliminar líneas de índice
    return text.strip()


## Funciones que controlan el flujo de extracción del texto.

In [ ]:
def detect_language(text: str) -> str:
    """
    Detecta el idioma de un texto.
    Devuelve código ISO 639-1 ('es', 'en', etc.)
    o 'unknown' si no hay suficiente texto.
    """
    try:
        # Necesita mínimo ~20 chars para ser fiable
        if len(text.strip()) < 20:
            return "unknown"
        return detect(text)
    except LangDetectException:
        return "unknown"

def extract_text_by_prompt(page, client: OpenAI, prompting_strategy: Dict) -> str:
  """
  page: página del documento pdf.
  client: Instancia de cliente de OpenAI.
  prompting_strategy: diccionario con la información relevante necesaria
                      para llevar a cabo la consulta contra la API

  return: Retorna el texto de la respuesta del modelo.
  """

  image_base64 = base64.b64encode(page.get_pixmap(dpi = 300).tobytes("png")).decode("utf-8")
  model = prompting_strategy.get("model", "gpt-4")
  prompt = prompting_strategy.get("prompt",
        "The user will provide you an image of a document file. Perform the following actions: "
        "1. Transcribe the text on the page. **TRANSCRIPTION OF THE TEXT:** "
        "2. If there is a chart, describe the image and include the text **DESCRIPTION OF THE IMAGE OR CHART** "
        "3. If there is a table, transcribe the table and include the text **TRANSCRIPTION OF THE TABLE**"
        "Transcribe the full content of this document page as plain text."
        "Follow these rules strictly:"
        "1. Transcribe all visible text exactly as it appears, preserving the logical reading order."
        "2. Ignore and do not transcribe: page headers, page footers, watermarks, page numbers, and any text that appears as a repeated document title or version label."
        "3. If the page contains a table, transcribe its content row by row, separating columns with a pipe character (|)."
        "4. If the page contains a chart or graph, describe its type, axes, legend and main values in a single paragraph starting with [CHART]:"
        "5. If the page contains a decorative image with no informational value, write [IMAGE: non-informative] and skip it."
        "6. Do not add explanations, comments or formatting such as Markdown."
        "7. If the page is blank or has no extractable content, return [EMPTY PAGE]."
    )

  response = client.chat.completions.create(
      model= model,
      temperature=0,
      messages=[
          {
              "role": "user",
              "content": [
                  {"type": "text", "text": prompt},
                  {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{image_base64}"}}
              ]
          }
      ]
  )
  text = response.choices[0].message.content
  return text


def extract_text_page(page, client: OpenAI, prompting_strategy: Dict) -> Tuple[str, str]:
  """
  page: página del documento pdf.
  client: Instancia de cliente de OpenAI.
  prompting_strategy: diccionario con la información relevante necesaria
                      para llevar a cabo la consulta contra la API.

  return: Retorna el texto de la página (str) y el método de extracción (str).
  """
  #Si pymupdf detecta que hay una tabla en la página, extraemos toda la página con
  #la api de openAI.
  if page.find_tables().tables:
    return extract_text_by_prompt(page, client, prompting_strategy), "openai_vision"

  #Si no se detecta la tabla extraemos el texto directamente. Ese texto obtenido
  #se debe comprobar entes de devolverlo por si hubiera habido alguna complicación
  # en la extracción que hubiera dado lugar a caracteres extraños o mal formateados.
  #En caso de que la extracción del texto no hubiese sido exitosa, se manda a ser
  #extraido vía API.
  text = page.get_text("text")
  if is_text_usable(text):
        return text, "pymupdf"

  return extract_text_by_prompt(page, client, prompting_strategy), "openai_vision"



def extract_text_block(file_path: str, client: OpenAI, prompting_strategy: Dict)-> List[Dict]:
    """
    file_path: Ruta al archivo pdf.
    client: Instancia de cliente de OpenAI.
    prompting_strategy: diccionario con la información relevante necesaria
                        para llevar a cabo la consulta contra la API.
    return: Retorna una lista de página, donde cada página es un diccionario.
    """
    paginas = []
    doc = fitz.open(file_path)  #Abrir el fichero con pymupdf

    #Iterar por todas las páginas del fichero, para determinar la forma que se
    #va a emplear para la extracción del texto (pymupdf o OpenAI API)
    for page_num, page in enumerate(doc, start=1):
        text, method = extract_text_page(page, client, prompting_strategy)
        texto_procesado = clean_page_text(text)

        paginas.append({"num_pagina":page_num,
                        "metodo_extraccion": method,
                        "texto_crudo":text,
                        "texto_procesado": texto_procesado#, "idioma": detect_language(texto_procesado)
                        })

    doc.close()
    return paginas

## Funciones para quitar los pies y encabezados de página.

Un header/footer recurrente — texto que el PDF tiene embebido como elemento de cabecera o pie en cada página. En el caso del EINF de AMC, el texto "Feeding the future" es el eslogan corporativo que aparece como watermark/header en prácticamente todas las páginas, y "ESTADO DE INFORMACIÓN NO FINANCIERA/ CONSOLIDADO DEL EJERCICIO: 2022 – 2023" es el título del documento repetido como cabecera.
​

¿Deberías Quitarlos?
Sí, para el RAG es claramente beneficioso eliminarlos. La razón es concreta: si ese texto aparece en 50 páginas, tu vector store tendrá 50 chunks con esa frase. Cuando el retriever busque por similitud semántica, esos chunks "contaminados" aparecerán como relevantes para queries que no tienen nada que ver, introduciendo ruido en el contexto que recibe el LLM y aumentando el riesgo de alucinación.

Cómo Detectarlos y Eliminarlos
La estrategia más robusta es detección estadística por frecuencia: si una línea aparece en más de un porcentaje umbral de las páginas de un documento, (se repite mucho), y además esa línea aparece al principio o al final del documento, podemos asumir que la línea en cuestión se trata de una cabecera o de un pie de página.

**No se hace jsuto al obtener el texto de la página, por que para poder idnetificar las cabeceras y los pies de página, necesito tener una foto global de todo el documento**

In [ ]:
from collections import Counter



def detect_repeated_headers_footers(pages: list, threshold: float = 0.30, n_lines: int = 5) -> set:
    """
    Detecta cabeceras y pies de página recurrentes combinando
    dos señales: frecuencia entre páginas Y posición en la página.

    Args:
        pages: Lista de dicts con campo 'texto_procesado'.
        threshold: Fracción mínima de páginas en que debe aparecer
                   una línea en posición de cabecera/pie. Default: 30%.
        n_lines: Número de líneas a considerar como zona de
                 cabecera (primeras N) y pie (últimas N). Default: 3.

    Returns:
        Conjunto de líneas identificadas como cabeceras/pies.
    """
    total_pages = len(pages)
    if total_pages == 0:
        return set()

    lines_counter = Counter()

    for page in pages:
        lines = [
            line.strip()
            for line in page["texto_procesado"].splitlines()
            if line.strip()
        ]
        if not lines:
            continue

        # Zona de cabecera: primeras N líneas
        header_zone = set(lines[:n_lines]) #Seleccionamos como cabecera de la página solo las N primeras líneas

        # Zona de pie: últimas N líneas
        footer_zone = set(lines[-n_lines:]) #Seleccionamos como pie de página solo las N últimas líneas

        border_zone = header_zone | footer_zone #Unimos ambas zonas
        lines_counter.update(border_zone)



    # Una línea es cabecera/pie si aparece en la cabecera o el pie en al menos el threshold% de las páginas
    min_count = total_pages * threshold
    repeated = {
        line for line, count in lines_counter.items()
        if count >= min_count
    }

    if repeated:
        print(f"{len(repeated)} cabecera(s)/pie(s) detectados:")
        for line in repeated:
            print(f"   → '{line}'")

    return repeated

def remove_headers_footers(pages: list, repeated_lines: set) -> List:
    """
    Elimina las líneas recurrentes del texto de cada página.
    """
    cleaned_pages = []
    for page in pages:
        clean_lines = [
            line for line in page["texto_procesado"].splitlines()
            if line.strip() not in repeated_lines
        ]
        cleaned_page = page.copy() #Hacemos una copia ya que al tratarse de una lista de diccionarios, estos se pasan por referencia, es decri no son copias
        cleaned_page["texto_procesado"] = "\n".join(clean_lines).strip()
        cleaned_pages.append(cleaned_page)

    return cleaned_pages


## Función que desencadena el flujo de procesamiento de cada documento.

In [ ]:
def begin_process(config: Dict):
    """
    config: Diccionario con la configuración del proceso.

    """
    #Extracción de la información de la configuración
    print("Iniciando listado de archivos...")
    input_conf = config.get("input", {})
    output_conf = config.get("output", {})
    base_path = input_conf.get("path")
    final_path = output_conf.get("path")

    file_patterns = input_conf.get("file_pattern", [])

    #Recogemos la estrategia de prompt que se usará para extraer el texto aplicando la api de OpenAI
    prompting_strategy = config.get("llm_model", {})
    token = prompting_strategy.get("key", "")


    next_file_number = 1  #Inicializamos el ID que tendrá el documento de salida
    os.makedirs(final_path, exist_ok=True) #Creamos el directorio final al que irán los jsons procesados
    client = OpenAI(api_key=token) #Instanciamos un único cliente para todas la peticiones a la API

    if not base_path:
        raise ValueError("No se ha especificado la ruta de entrada en la configuración.")

    #Obtenemos todos los ficheros que vamos a procesar
    matched_files = []
    try:
      for file in sorted(os.listdir(base_path)):
        full_path = os.path.join(base_path, file)
        #Comprobamos que realmente sea un archivo de las extensiones esperadas según nuestro archivo de configuración.
        if os.path.isfile(full_path) and any(file.lower().endswith(pattern.lower()) for pattern in file_patterns):
            matched_files.append(full_path)

    except Exception as e:
        error_msg = f"Error al listar archivos en {base_path}: {e}"
        return

    print(f"Se han encontrado {len(matched_files)} archivos.")


    current_date = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    for i, file_path in enumerate(matched_files, start = 1):
        file_name = os.path.basename(file_path)
        print(f"\n [{i}/{len(matched_files)}] Procesando: {file_name}")

        try:
            #Comenzamos a rellenar el fichero json final de salida con los campos
            #informativos del documento (metadata)
            new_entry = {
                "nombre_original": file_name,
                "ruta": file_path,
                "nombre_procesado": f'D-{next_file_number}-{file_name}',
                "fecha_procesado": current_date
            }
            print(f"Procesando archivo: {file_name}")


            pages_list = extract_text_block(file_path, client, prompting_strategy)
            repeated = detect_repeated_headers_footers(pages_list) if len(pages_list) > 1 else set()

            if repeated:
                pages_list = remove_headers_footers(pages_list, repeated)

            new_entry["num_paginas"] = len(pages_list)
            new_entry["paginas"] = pages_list
            new_entry["texto_completo"] = "\n\n".join(
                                                      p["texto_procesado"] for p in pages_list
                                                      if p["texto_procesado"] and p["texto_procesado"].strip() != "[EMPTY PAGE]" )# and p["idioma"] == "es") #in ("es", "unknown")



        except Exception as e:
            error_msg = f"Error al procesar el archivo {file_path}: {e}"
            print(f"{error_msg}")
            new_entry = {
                "nombre_original": file_name,
                "ruta": file_path,
                "nombre_procesado": f'D-{next_file_number}',
                "fecha_procesado": current_date,
                "incidencia": error_msg,
            }
        finally:
            next_file_number += 1


        ruta_final = write_json_to_disk(final_path, new_entry)

        print(f"Fichero {new_entry['nombre_procesado']} guardado en {ruta_final}")


## Main()


In [ ]:
def main():
  """
  Función principal que ejecuta el flujo de procesamiento.
  """
  #Se obtiene la configuración vía un fichero yaml en local
  config = load_config("/content/drive/MyDrive/Colab Notebooks/Cuarto curso/TFG/Procesamiento-Corpus/cnf.yaml")
  begin_process(config)


if __name__ == "__main__":
    main()


## Función para visualizar el campo 'texto_completo' de los json

In [ ]:
def show_final_files_detail(
    output_path: str = "/content/drive/MyDrive/Colab Notebooks/Cuarto curso/TFG/Procesamiento-Corpus/jsons_output",
    file_prefix: Optional[str] = None
) -> None:
    """
    Muestra el texto de cada página individualmente,
    junto con el método de extracción usado.

    Args:
        output_path: Ruta al directorio con los JSON procesados.
        file_prefix: Si se especifica (ej. 'D-1'), solo muestra documentos
                     cuyo nombre empiece por ese prefijo (ej. 'D-1-nombre.json').
                     Si es None, muestra todos los documentos.
    """

    json_files = [
        f for f in os.listdir(output_path)
        if f.endswith(".json")
        and (file_prefix is None or f.startswith(f"{file_prefix}-"))
    ]

    if not json_files:
        msg = f"No se encontró ningún fichero con prefijo '{file_prefix}'" if file_prefix \
              else "No se encontraron ficheros JSON en el directorio de salida."
        print(f" {msg}")
        return

    for json_file in sorted(json_files):
        file_path = os.path.join(output_path, json_file)

        with open(file_path, "r", encoding="utf-8") as f:
            data = json.load(f)

        print("=" * 80)
        print(f" {data.get('nombre_original')} — {data.get('num_paginas')} páginas")
        print("=" * 80)

        for page in data.get("paginas", []):
            print(f"\n--- Página {page['num_pagina']} [{page['metodo_extraccion']}] ---")
            print(page.get("texto_procesado", ""))

        print("\n")


Documento 1

In [ ]:

show_final_files_detail(file_prefix="D-1")



 EINF-2022_2023-08.09.24.pdf — 33 páginas

--- Página 1 [pymupdf] ---
ESTADO DE INFORMACIÓN NO FINANCIERA
CONSOLIDADO DEL EJERCICIO: 2022 - 2023 
AMC GLOBAL, S.L.U. y sociedades dependientes

--- Página 2 [pymupdf] ---
1. Conoce AMC GLOBAL.
1.1 Carta presidencia AMC GLOBAL
1.2 Marco Normativo
1.3 AMC GLOBAL ¿Quiénes somos? 
1.4 Materialidad
2. AMC GLOBAL con el medioambiente.
2.1 Desarrollo sostenible (Taxonomía)
2.2 Recursos naturales
2.3 Contaminación y Cambio Climático
2.4 Tratamiento sostenible de los residuos
3. AMC GLOBAL y sus personas.
3.1 Capital humano
3.2 Igualdad
3.3 Salud en el empleo
3.4 AMC GLOBAL y sus clientes y consumidores
3.5 AMC GLOBAL y su cadena de suministro
3.6 AMC GLOBAL en la sociedad
4. AMC GLOBAL y su forma de gobierno.
4.1 Gobierno corporativo
4.2 Derechos Universales
ANEXO: Tabla de contenidos de la Ley 11/2018 40
 
05 - 11

06 - 07

09 -11
12 - 25
12 - 14
15 - 18
19 - 21
22 - 25
26 - 55
26 - 37
38 - 40
40 - 42
43 - 45
46 - 47
48 - 55
56 - 61
56 - 58
58 -

In [ ]:
import json

file_path = "/content/drive/MyDrive/Colab Notebooks/Cuarto curso/TFG/Procesamiento-Corpus/jsons_output/D-10-POL-CALI-9.0 V1_Politica de Calidad y Seguridad Alimentaria.pdf.json"

with open(file_path, "r", encoding="utf-8") as f:
    data = json.load(f)

data["texto_completo"] = data["texto_completo"].replace("POL-CALI-9.0 \nRevisión 01\n© AMC Natural Drinks Group\n", "")
for page in data.get("paginas", []):
    page["texto_procesado"] = page["texto_procesado"].replace("POL-CALI-9.0 \nRevisión 01\n© AMC Natural Drinks Group\n", "")
with open(file_path, "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=4)

print("Sustitución completada.")
